# Notebook 01 â€” Dataset Inspection

**Goal:** Understand the structure of the uploaded CSV dataset before any cleaning or modelling.

We inspect:
- total file count
- filename patterns
- metadata structure
- column names per cycle type
- battery IDs / cycle counts
- data types and missing values

> **Nothing is assumed.  Everything is determined from the actual files.**

In [ ]:
import os, sys, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('..', 'src'))
from data_loader import load_metadata, parse_start_time

# Paths (relative to notebooks/)
DATASET_DIR  = os.path.join('..', 'dataset', 'data')
METADATA_PATH = os.path.join('..', 'dataset', 'metadata.csv')
print('Imports OK')

## 1. File System Overview

In [ ]:
# Count all CSV files in dataset/data/
all_files = [f for f in os.listdir(DATASET_DIR) if f.endswith('.csv')]
all_files.sort()
print(f"Total CSV files in dataset/data/  : {len(all_files)}")
print(f"First file : {all_files[0]}")
print(f"Last file  : {all_files[-1]}")
print()

# Check naming pattern
sample = all_files[:10]
print("Sample filenames:", sample)
pattern_ok = all(re.match(r'^\d{5}\.csv$', f) for f in all_files)
print(f"\nAll filenames match 5-digit pattern: {pattern_ok}")

## 2. Metadata Inspection

In [ ]:
meta = load_metadata(METADATA_PATH)
print(f"metadata.csv shape: {meta.shape}")
print(f"Columns: {list(meta.columns)}")
print()
print(meta.dtypes)
print()
print("First 5 rows:")
meta.head()

In [ ]:
# Type distribution
print("Cycle type distribution:")
print(meta['type'].value_counts())
print()
print("Missing values per column:")
print(meta.isnull().sum())

In [ ]:
# Battery IDs
batteries = sorted(meta['battery_id'].unique())
print(f"Number of unique batteries: {len(batteries)}")
print(f"Battery IDs: {batteries}")

In [ ]:
# Discharge cycles per battery
dis = meta[meta['type'] == 'discharge']
counts = dis.groupby('battery_id').size().sort_values(ascending=False)
print(f"Total discharge cycles: {len(dis)}")
print()
print("Discharge cycles per battery:")
print(counts.to_string())

## 3. Sample File Inspection

In [ ]:
# â”€â”€ Discharge sample â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
dis_sample = dis.iloc[0]
dis_file = os.path.join(DATASET_DIR, dis_sample['filename'])
df_dis = pd.read_csv(dis_file)
print(f"=== Discharge sample: {dis_sample['filename']} ===")
print(f"  Battery: {dis_sample['battery_id']}  |  Capacity: {dis_sample['Capacity']}")
print(f"  Rows: {len(df_dis)}  |  Columns: {list(df_dis.columns)}")
print(df_dis.head(3).to_string())
print()
print("Data types:")
print(df_dis.dtypes)

In [ ]:
# â”€â”€ Charge sample â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
chg = meta[meta['type'] == 'charge']
chg_file = os.path.join(DATASET_DIR, chg.iloc[0]['filename'])
df_chg = pd.read_csv(chg_file)
print(f"=== Charge sample: {chg.iloc[0]['filename']} ===")
print(f"  Rows: {len(df_chg)}  |  Columns: {list(df_chg.columns)}")
print(df_chg.head(3).to_string())

In [ ]:
# â”€â”€ Impedance sample â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
imp = meta[meta['type'] == 'impedance']
imp_file = os.path.join(DATASET_DIR, imp.iloc[0]['filename'])
df_imp = pd.read_csv(imp_file)
print(f"=== Impedance sample: {imp.iloc[0]['filename']} ===")
print(f"  Rows: {len(df_imp)}  |  Columns: {list(df_imp.columns)}")
print(df_imp.head(3).to_string())

## 4. Row-count Distribution for Discharge Files

In [ ]:
# Sample 50 discharge files and check their row counts
sample_dis = dis.sample(min(50, len(dis)), random_state=42)
row_counts = []
for _, row in sample_dis.iterrows():
    fp = os.path.join(DATASET_DIR, row['filename'])
    try:
        n = sum(1 for _ in open(fp)) - 1  # fast line count
        row_counts.append(n)
    except Exception:
        pass

rc = pd.Series(row_counts)
print("Row count statistics for 50 sampled discharge CSVs:")
print(rc.describe().round(0).astype(int).to_string())

## 5. Timestamp Parsing Check

In [ ]:
# Parse start_time for all metadata rows
print("Sample raw start_time values:")
for v in meta['start_time'].head(5):
    parsed = parse_start_time(v)
    print(f"  {repr(v)[:60]}  ->  {parsed}")

parsed_all = meta['start_time'].apply(parse_start_time)
n_parsed = parsed_all.notna().sum()
print(f"\nSuccessfully parsed: {n_parsed} / {len(meta)}")

## 6. Dataset Inspection Report

In [ ]:
print("=" * 55)
print("  DATASET INSPECTION REPORT")
print("=" * 55)
print(f"  Total CSV files in dataset/data/     : {len(all_files)}")
print(f"  Rows in metadata.csv                 : {len(meta)}")
print(f"  Cycle types:")
for t, n in meta['type'].value_counts().items():
    print(f"    {t:15s} : {n}")
print(f"  Unique batteries                     : {len(batteries)}")
print(f"  Discharge cycles (potential samples) : {len(dis)}")
print(f"  Capacity available for all discharge : {dis['Capacity'].notna().all()}")
print()
print("  Discharge CSV columns:")
for c in df_dis.columns:
    print(f"    {c}")
print()
print("  Key finding: ONE CSV = ONE BATTERY OPERATION")
print("  Each discharge CSV is a complete, independent discharge cycle.")
print("  Capacity is pre-computed in metadata.csv.")
print("=" * 55)